In this assignment, we are given a task to predict the position of a pixel in an image.
Each image is of size 50×50 and contains only one bright pixel (value 255) while all other pixels are 0.
The goal is to train a model that can predict the (x, y) coordinates of that bright pixel.

In [13]:
import numpy as np

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import layers, models

## Dataset Creation
we create the dataset by going through all possible positions in a 50×50 image.
for each position, we place a bright pixel (255) at that location and keep all other pixels 0.
this gives us 50 × 50 = 2500 images.


In [14]:
IMG_SIZE = 50

X = []
y = []

for i in range(IMG_SIZE):
    for j in range(IMG_SIZE):

        img = np.zeros((IMG_SIZE, IMG_SIZE))
        img[i, j] = 255

        X.append(img)
        y.append([i, j])

X = np.array(X)
y = np.array(y)

# normalize input
X = X / 255.0

# reshape for CNN
X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2500, 50, 50, 1)
y shape: (2500, 2)


## Train, Validation and Test Split
We split the dataset into training, validation and testing sets.
Training data is used to train the model, validation data is used to check performance during training, and test data is used for final evaluation.

In [15]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (2000, 50, 50, 1)
Val: (250, 50, 50, 1)
Test: (250, 50, 50, 1)


## Model (CNN)
We use a CNN because it works well with image data.
The model takes the image as input and predicts the (x, y) coordinates.

In [16]:
model = models.Sequential([
    layers.Conv2D(16, (3,3), activation='relu', input_shape=(50,50,1)),
    layers.MaxPooling2D(2,2),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(2)
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 48, 48, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 24, 24, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 22, 22, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 11, 11, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3872)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │       247,872 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 252,802 (987.51 KB)

 Trainable params: 252,802 (987.51 KB)

 Non-trainable params: 0 (0.00 B)

## Model Training
We train the model using training data and validate it using validation data.

In [17]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=32
)

Epoch 1/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - loss: 356.2250 - mae: 15.4665 - val_loss: 199.2243 - val_mae: 12.0533
Epoch 2/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 42ms/step - loss: 183.8021 - mae: 11.5647 - val_loss: 142.7783 - val_mae: 9.8977
Epoch 3/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 6s 63ms/step - loss: 84.3378 - mae: 6.6871 - val_loss: 46.3199 - val_mae: 4.2371
Epoch 4/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - loss: 37.1616 - mae: 3.8265 - val_loss: 34.0871 - val_mae: 3.7100
Epoch 5/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - loss: 27.4317 - mae: 3.3244 - val_loss: 26.9857 - val_mae: 3.1922
Epoch 6/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - loss: 21.7691 - mae: 2.9680 - val_loss: 22.3521 - val_mae: 2.9812
Epoch 7/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 52ms/step - loss: 18.3246 - mae: 2.8396 - val_loss: 19.5880 - val_mae: 2.8903
Epoch 8/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - loss: 15.7383 - mae: 2.6909 - val_loss: 19.9517 - val_mae: 3.0398
Epoch 9/10
63/63 ━━━━━━━━━━━━━━━━━━━━ 3s 

## Model Evaluation
We evaluate the model on test data to see how well it predicts the coordinates.

In [18]:
loss, mae = model.evaluate(X_test, y_test)
print("Test MAE:", mae)

8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 12.9577 - mae: 2.4977
Test MAE: 2.4977169036865234


## Conclusion
The model is able to predict the pixel coordinates with good accuracy.
This shows that CNN can learn spatial patterns from images and can be used for coordinate prediction tasks.